# Fine-tune the ScholarCrew writer with LoRA

Trains a small adapter on top of an open model, using the checked examples built by `scholarcrew.finetune.build_dataset`.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom.

In [7]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [8]:
# 2. Get the code and the libraries
!git clone -q https://github.com/omsawant-ds/scholarcrew.git
%cd scholarcrew
!pip install -q -U transformers peft accelerate bitsandbytes

/content/scholarcrew/scholarcrew


In [9]:
# 3. Upload your training examples: choose data/sft/train.jsonl from your computer.
#    The file stays in this session. It is not on GitHub.
import os
from google.colab import files

os.makedirs('data/sft', exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
os.replace(name, 'data/sft/train.jsonl')
!wc -l data/sft/train.jsonl

Saving train.jsonl to train.jsonl
103 data/sft/train.jsonl


In [10]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=src python -m scholarcrew.finetune.train_lora --model Qwen/Qwen3-8B --out outputs/writer-lora --batch-size 1 --grad-accum 16 --max-len 3072

examples: 93 train, 10 held out, 0 too long and skipped
Loading weights: 100% 399/399 [01:05<00:00,  6.06it/s]
trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
  0% 0/18 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[W1113 06:08:39.741657560 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2235564032 bytes (free: 427622400, total: 15637086208).
[W1113 06:08:46.423238022 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while

In [11]:
# 5. Download the adapter (a few hundred MB), so it survives the session ending.
!zip -qr writer-lora.zip outputs/writer-lora -x 'outputs/writer-lora/checkpoints/*'
from google.colab import files
files.download('writer-lora.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# 6. Optional: also keep a private copy on Hugging Face (free).
#    Create a token at huggingface.co/settings/tokens with 'write' access, then run:
# from huggingface_hub import notebook_login
# notebook_login()
# !PYTHONPATH=src python -m scholarcrew.finetune.train_lora --model Qwen/Qwen3-8B --out outputs/writer-lora --epochs 3 --push-to-hub YOUR-USERNAME/scholarcrew-writer-lora